# PyTorch Introduction

Learn the fundamentals of PyTorch for deep learning.

## Learning Objectives

- Understand PyTorch tensors and operations
- Learn automatic differentiation with autograd
- Build neural networks with nn.Module
- Train models with optimizers and loss functions
- Use DataLoader for batch processing

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Tensors: The Building Blocks

Tensors are multi-dimensional arrays, similar to NumPy arrays but with GPU support.

In [ ]:
# Creating tensors
print("=== Creating Tensors ===")

# From Python list
t1 = torch.tensor([1, 2, 3, 4])
print(f"From list: {t1}")

# From NumPy array
arr = np.array([[1, 2], [3, 4]])
t2 = torch.from_numpy(arr)
print(f"From NumPy: {t2}")

# Special tensors
zeros = torch.zeros(3, 4)
ones = torch.ones(2, 3)
rand = torch.rand(2, 3)  # Uniform [0, 1)
randn = torch.randn(2, 3)  # Standard normal

print(f"\nZeros (3x4):\n{zeros}")
print(f"\nRandom (2x3):\n{rand}")

In [ ]:
# Tensor properties
t = torch.randn(3, 4, 5)

print(f"Shape: {t.shape}")
print(f"Data type: {t.dtype}")
print(f"Device: {t.device}")
print(f"Number of dimensions: {t.ndim}")
print(f"Total elements: {t.numel()}")

In [ ]:
# Tensor operations
a = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
b = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32)

print("=== Element-wise Operations ===")
print(f"a + b:\n{a + b}")
print(f"a * b (element-wise):\n{a * b}")
print(f"a ** 2:\n{a ** 2}")

print("\n=== Matrix Operations ===")
print(f"a @ b (matrix multiply):\n{a @ b}")
print(f"a.T (transpose):\n{a.T}")

In [ ]:
# Reshaping tensors
t = torch.arange(12)
print(f"Original: {t}")

# Various reshapes
print(f"\nReshaped (3x4):\n{t.view(3, 4)}")
print(f"\nReshaped (2x2x3):\n{t.view(2, 2, 3)}")
print(f"\nUsing -1 for auto (4x-1):\n{t.view(4, -1)}")

## 2. Autograd: Automatic Differentiation

PyTorch tracks operations on tensors to compute gradients automatically.

In [ ]:
# Basic autograd example
x = torch.tensor([2.0, 3.0], requires_grad=True)

# Forward pass
y = x ** 2 + 3 * x + 1  # y = x^2 + 3x + 1
z = y.sum()

print(f"x: {x}")
print(f"y = x^2 + 3x + 1: {y}")
print(f"z = sum(y): {z}")

# Backward pass
z.backward()

# dy/dx = 2x + 3
print(f"\nGradient (dz/dx = 2x + 3): {x.grad}")
print(f"Expected for x=[2,3]: {2*x.detach() + 3}")

In [ ]:
# Computational graph visualization concept
print("Computational Graph:")
print("")
print("  x (leaf, requires_grad=True)")
print("  │")
print("  ▼")
print("  y = x² + 3x + 1 (intermediate)")
print("  │")
print("  ▼")
print("  z = sum(y) (output)")
print("  │")
print("  ▼")
print("  z.backward()  →  x.grad = dz/dx")

In [ ]:
# Gradient accumulation and zeroing
x = torch.tensor([1.0], requires_grad=True)

for i in range(3):
    y = x * 2
    y.backward()
    print(f"Iteration {i+1}: x.grad = {x.grad}")

print("\n(Gradients accumulate!)")
print("\nWith zeroing:")

x = torch.tensor([1.0], requires_grad=True)
for i in range(3):
    if x.grad is not None:
        x.grad.zero_()
    y = x * 2
    y.backward()
    print(f"Iteration {i+1}: x.grad = {x.grad}")

## 3. Building Neural Networks with nn.Module

In [ ]:
# Simple neural network using nn.Sequential
simple_net = nn.Sequential(
    nn.Linear(2, 16),   # Input: 2 features, Hidden: 16 neurons
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

print(simple_net)

# Test forward pass
x = torch.randn(5, 2)
output = simple_net(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Output values: {output.flatten()}")

In [ ]:
# Custom neural network class
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(NeuralNet, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Create network
model = NeuralNet(input_size=2, hidden_sizes=[16, 8], output_size=1)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

## 4. Loss Functions and Optimizers

In [ ]:
# Common loss functions
print("=== Loss Functions ===")

# Binary Cross Entropy
bce_loss = nn.BCELoss()
y_pred = torch.tensor([0.9, 0.1, 0.8])
y_true = torch.tensor([1.0, 0.0, 1.0])
print(f"BCE Loss: {bce_loss(y_pred, y_true):.4f}")

# Mean Squared Error
mse_loss = nn.MSELoss()
print(f"MSE Loss: {mse_loss(y_pred, y_true):.4f}")

# Cross Entropy (for multi-class)
ce_loss = nn.CrossEntropyLoss()
logits = torch.tensor([[2.0, 0.5, 0.1], [0.1, 2.5, 0.3]])
labels = torch.tensor([0, 1])
print(f"Cross Entropy Loss: {ce_loss(logits, labels):.4f}")

In [ ]:
# Common optimizers
print("=== Optimizers ===")

model = NeuralNet(2, [16, 8], 1)

# SGD
sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
print(f"SGD: {sgd}")

# Adam (most popular)
adam = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))
print(f"\nAdam: {adam}")

# AdamW (with weight decay)
adamw = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
print(f"\nAdamW: {adamw}")

## 5. DataLoader for Batch Processing

In [ ]:
# Create dataset
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

# Create DataLoader
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# Iterate through batches
print("Sample batches:")
for i, (batch_x, batch_y) in enumerate(train_loader):
    print(f"Batch {i+1}: X shape = {batch_x.shape}, y shape = {batch_y.shape}")
    if i >= 2:
        print("...")
        break

## 6. Complete Training Loop

In [ ]:
def train_model(model, train_loader, test_loader, epochs=100, lr=0.01):
    """
    Complete training loop.
    
    Parameters
    ----------
    model : nn.Module
        Neural network model
    train_loader : DataLoader
        Training data loader
    test_loader : DataLoader
        Test data loader
    epochs : int
        Number of training epochs
    lr : float
        Learning rate
    
    Returns
    -------
    dict : Training history
    """
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for batch_x, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            predicted = (outputs > 0.5).float()
            train_correct += (predicted == batch_y).sum().item()
            train_total += batch_y.size(0)
        
        # Evaluation phase
        model.eval()
        test_loss = 0
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                
                test_loss += loss.item()
                predicted = (outputs > 0.5).float()
                test_correct += (predicted == batch_y).sum().item()
                test_total += batch_y.size(0)
        
        # Record history
        history['train_loss'].append(train_loss / len(train_loader))
        history['test_loss'].append(test_loss / len(test_loader))
        history['train_acc'].append(train_correct / train_total)
        history['test_acc'].append(test_correct / test_total)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] - "
                  f"Train Loss: {history['train_loss'][-1]:.4f}, "
                  f"Train Acc: {history['train_acc'][-1]:.4f}, "
                  f"Test Acc: {history['test_acc'][-1]:.4f}")
    
    return history

# Train the model
model = NeuralNet(2, [16, 8], 1)
history = train_model(model, train_loader, test_loader, epochs=100, lr=0.01)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['test_loss'], label='Test Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Test Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Accuracy')
axes[1].plot(history['test_acc'], label='Test Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Test Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Plot decision boundary
def plot_pytorch_decision_boundary(model, X, y, scaler, title='PyTorch Decision Boundary'):
    model.eval()
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid)
    grid_tensor = torch.FloatTensor(grid_scaled)
    
    with torch.no_grad():
        Z = model(grid_tensor).numpy().reshape(xx.shape)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, levels=50, cmap='RdYlBu', alpha=0.8)
    plt.colorbar(label='Probability')
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    
    X_orig = scaler.inverse_transform(X)
    plt.scatter(X_orig[y.flatten()==0, 0], X_orig[y.flatten()==0, 1], c='blue', edgecolors='black', label='Class 0')
    plt.scatter(X_orig[y.flatten()==1, 0], X_orig[y.flatten()==1, 1], c='red', edgecolors='black', label='Class 1')
    plt.legend()
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

# Use original unscaled X for plotting
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    *make_moons(n_samples=1000, noise=0.2, random_state=42), test_size=0.2, random_state=42
)
plot_pytorch_decision_boundary(model, X_train, y_train_t.numpy(), scaler)

## 7. Saving and Loading Models

In [ ]:
# Save model
torch.save(model.state_dict(), '/tmp/model_weights.pth')
print("Model weights saved!")

# Save entire model (not recommended for production)
torch.save(model, '/tmp/full_model.pth')
print("Full model saved!")

In [ ]:
# Load model weights
new_model = NeuralNet(2, [16, 8], 1)
new_model.load_state_dict(torch.load('/tmp/model_weights.pth', weights_only=True))
new_model.eval()

# Verify loaded model
with torch.no_grad():
    original_output = model(X_test_t[:5])
    loaded_output = new_model(X_test_t[:5])

print("Original model output:", original_output.flatten().numpy())
print("Loaded model output:  ", loaded_output.flatten().numpy())
print("Outputs match:", torch.allclose(original_output, loaded_output))

## 8. Common PyTorch Patterns

In [ ]:
# Pattern 1: Model with Dropout and BatchNorm
class RobustNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout=0.2):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.dropout1 = nn.Dropout(dropout)
        
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.bn2 = nn.BatchNorm1d(hidden_size // 2)
        self.dropout2 = nn.Dropout(dropout)
        
        self.fc3 = nn.Linear(hidden_size // 2, output_size)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        return torch.sigmoid(x)

robust_model = RobustNet(2, 32, 1, dropout=0.2)
print(robust_model)

In [ ]:
# Pattern 2: Learning rate scheduling
model = NeuralNet(2, [16, 8], 1)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Step scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

# Or reduce on plateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=10, factor=0.5)

print("Learning rate schedulers configured!")

## 9. Key Takeaways

1. **Tensors** are like NumPy arrays but with GPU support and autograd
2. **Autograd** automatically computes gradients for backpropagation
3. **nn.Module** is the base class for all neural networks
4. **DataLoader** handles batching and shuffling efficiently
5. **Training loop**: forward → loss → backward → optimizer step
6. Always zero gradients before backward pass
7. Use `model.train()` and `model.eval()` to toggle dropout/batchnorm behavior
8. Use `torch.no_grad()` during inference to save memory

In [ ]:
# Summary comparison: NumPy vs PyTorch
import pandas as pd

comparison = pd.DataFrame({
    'Operation': ['Create array', 'Matrix multiply', 'Reshape', 'GPU support', 'Autograd'],
    'NumPy': ['np.array()', 'a @ b', 'a.reshape()', 'No', 'No'],
    'PyTorch': ['torch.tensor()', 'a @ b', 'a.view()', 'Yes (.cuda())', 'Yes (requires_grad)']
})
print(comparison.to_string(index=False))